# Adding and splitting sectors

This notebook uses the packaged MARIO IOT test table to show the Excel workflow for extending a database with new sectors.

The screenshots below refer to the IOT workbook. SUT follows the same workflow, but the `Master` sheet uses `Activity` and `Commodity`, includes `Market share`, and does not expose `Add or Split`.

The `Split` workflow is for the moment supported only for the IOT case and relies on a cross-minimum entropy optimization model written in the [CVXlab](https://cvxlab.readthedocs.io/en/latest/) framework and embedded into MARIO. A publications on this model will be soon sent to peer review

In [ ]:
import mario

## Load the test table

We start from the packaged IOT test table so the workflow stays reproducible and easy to inspect.

In [ ]:
db = mario.load_test("IOT")
db.get_index("Sector")

___
## Generate an empty add-sector template

The add-sectors workbook is usually filled in two passes:

1. generate the empty workbook;
2. fill the `Master` sheet;
3. let MARIO create the missing inventory sheets;
4. fill the inventory sheets in Excel;
5. read the completed workbook and apply it.

The next sections explain what each sheet is for before running the workflow.

### Download the packaged example workbooks

The exact workbooks used in this example are available here:

- [Empty add-sector template](../../_static/data/supporting_files/add_sector_iot_template.xlsx)
- [Filled master workbook](../../_static/data/supporting_files/add_sector_iot_master_filled.xlsx)
- [Completed inventories workbook](../../_static/data/supporting_files/add_sector_iot_inventories_filled.xlsx)

In [ ]:
db.get_add_sectors_excel(
    path="/path/to/add_sector_template.xlsx",
    overwrite=True,
)

### The Master sheet

![The add-sectors master sheet](../../_static/images/add_sector_master.png)

The `Master` sheet defines the high-level structure of the extension:

- `Region`: target database region. You can also use a region cluster name to replicate the same characterization in multiple regions.
- `Sector`: name of the new sector.
- `Inventory sheet`: name of the sheet where the inventory is described.
- `Quantity` and `Unit`: functional unit of the new sector's output.
- `Final consumption` and `Consumption category`: optional final demand attached to the new sector.
- `Parent Sector`: existing sector used as the reference when coefficients or percentage changes should inherit an existing structure.
- `Leave empty`: when set to `TRUE`, MARIO skips that inventory sheet.
- `Add or Split`: use `Add` for a fully new sector, or `Split` when the new sector must be extracted from a parent sector.

The same inventory sheet name can be reused across multiple `Master` rows when one characterization should be applied to multiple target regions.

### The Clusters sheets

![The add-sectors cluster sheets](../../_static/images/add_sector_clusters.png)

Cluster sheets are optional, but they are useful whenever the same assumptions must be repeated:

- `Regions Clusters`: each column header is one cluster name, and each non-empty cell below it is one database region included in that cluster.
- `Sectors Clusters`: in IOT, this lets you group multiple existing sectors under one reusable label.
- `Commodities Clusters`: in SUT, the item-cluster sheet works in the same way for commodity inputs.

Region clusters can be referenced in `Master.Region` and in `Inventory.DB Region`. Item clusters can be referenced in `Inventory.DB Item`.

___
## Create inventory sheets from a filled Master sheet

Fill the `Master` sheet in Excel first. Once the rows and inventory-sheet names are ready, read the workbook back with `get_inventories=True` so MARIO creates the missing inventory tabs inside the same file.

In [ ]:
db.read_add_sectors_excel(
    path="/path/to/add_sector_master_filled.xlsx",
    get_inventories=True,
)

### The inventory sheets

![The add-sectors inventory sheet](../../_static/images/add_sector_inventories.png)

Each inventory sheet describes the inputs used by the new sector:

- `Quantity`: technical coefficient per unit of output declared in `Master`.
- `Unit`: unit of that coefficient.
- `Input`: free user label; MARIO does not use it as a database key.
- `Item type`: database set to target, such as `Sector`, `Factor of production`, or `Satellite account`.
- `DB Item`: existing database label, or an item-cluster name when clusters are used.
- `DB Region`: source region of the input. This can be a database region or a region cluster.
- `Change type`: use `Update` for an absolute coefficient, or `Percentage` to scale the parent structure.
- `Source` and `Notes`: support columns for references and comments.

The `DB units` sheet in the workbook is a reference tab that helps you keep units consistent with the database.

### Split-specific sheets

If at least one row in `Master` is marked as `Split`, MARIO also relies on `Total outputs`, `Trades`, (both mandatory) `Exclusions` (optional), and `Tolerances` (mandatory but prefilled).
For the moment, this is an IOT-only workflow. In practice, `Split` means that the new sector is extracted from an existing parent sector rather than added as a completely independent structure.


#### The Total outputs sheet

![The add-sectors total outputs sheet](../../_static/images/add_sector_total_outputs.png)

For each new sector, the user must specify the production level in each region of the *database*


#### The Trades sheet

![The add-sectors trades sheet](../../_static/images/add_sector_trades.png)

For each new sector in each region, the user must specify the amount of product imported by the other regions


#### The Exclusion sheet

Here you can specify whether some of those sectors indicated as `Sector_to` do not consume any input coming from `Sector_from`. In our simplified example, we did not implement any exclusion


#### The Tolerances sheet

Here you can modify the rigidity of the optimization problem by acting on the `delta` and `eps` parameters. 

## Read the completed workbook

After filling the generated inventory tabs and, if necessary, the split-specific data sheets, read the completed workbook with `read_inventories=True`. MARIO will normalize the `Master` sheet, load clusters, and group the inventories by target sector.

In [ ]:
db.read_add_sectors_excel(
    path="/path/to/add_sector_inventories_filled.xlsx",
    read_inventories=True,
)

db.inventories.keys()

## Apply the completed workbook

Once the inventories are loaded, `db.add_sectors(...)` inserts the new sectors into the database. 

In [ ]:
expanded_db = db.add_sectors(inplace=False)
expanded_db.get_index("Sector")

The updated coefficient and final-demand blocks now contain the new sectors.

In [ ]:
expanded_db.z

In [ ]:
expanded_db.Y

## Disaggregate a new sector from an existing one (with CVXLab)

By "splitting", we mean "disaggregating a new sector from an existing one". This workflow is for IOT only and is supported by a CVXlab model.

### Download the bundled split-model assets

For transparency, the exact CVXLab assets used by this workflow are available here:

- [Model formulation (pdf)](../../_static/data/add_sectors/Model%20formulation.pdf)
- [CVXlab model settings](../../_static/data/add_sectors/model_settings.xlsx)
- [Mapping file between MARIO and CVXlab](../../_static/data/add_sectors/mapping.xlsx)

In [ ]:
expanded_db_split = db.add_sectors(
    inplace=False,
    split=True,
    cvxlab_path="/path/to/cvxlab",
)

In [ ]:
expanded_db_split.Z